[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab4_GraphRAG_KnowledgeGraphs_Solutions.ipynb)


# Lab 4: GraphRAG — Knowledge Graphs for Clinical Knowledge (SOLUTIONS)
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Vector RAG is great for 'what is the dose for cisplatin?' but fails for 'what are the relationships between staging, histology, and treatment?' GraphRAG extracts entities and relationships, builds a knowledge graph, and answers multi-hop clinical questions.

### Objective
- Extract clinical entities and relationships from WT guidelines
- Build a knowledge graph with NetworkX
- Visualize the graph
- Query with graph traversal
- Compare with vector RAG on multi-hop questions


---
## Setup

Install packages and set Colab secrets (`OPENAI_API_KEY`, `LLAMA_CLOUD_API_KEY`).

**What is GraphRAG?**  
Standard vector RAG retrieves *chunks of text* close to your query. GraphRAG instead extracts *entities* (drugs, stages, diseases) and *relationships* (TREATS, CAUSES) into a graph, then answers questions by traversing edges — no embeddings needed for the reasoning step.

**Lab flow:**
1. Define a typed clinical ontology (entities + relationship types)
2. Load WT guideline text → split into ~8 coherent chunks
3. LLM with `structured_output` extracts a `GraphExtraction` object per chunk
4. Assemble entities/relationships into a NetworkX `DiGraph`
5. Visualize interactively with pyvis
6. Answer multi-hop questions via graph traversal
7. Detect thematic clusters with Louvain community detection
8. Compare with vector RAG on the same question

Provide **`/content/wt.md`** (saved from Lab 1) or the notebook loads it from Google Drive.

In [ ]:
!pip install -q networkx matplotlib langchain langchain-text-splitters langchain-openai openai pyvis langchain-community llama-parse faiss-cpu tiktoken

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('Loaded OPENAI_API_KEY and LLAMA_CLOUD_API_KEY from Colab secrets.')


## Cell 1 — Schemas

GraphRAG requires a **typed ontology** — a controlled vocabulary the LLM must map text into.

**Entity types** (clinical nouns):

| Type | Examples |
|------|---------|
| Disease | wilms tumor, neuroblastoma |
| Stage | stage iii, stage i |
| Drug | vincristine, dactinomycin, doxorubicin |
| SideEffect | nephrotoxicity, nausea, cardiotoxicity |
| Procedure | nephrectomy, radiation therapy |
| Histology | favorable histology, anaplastic |
| Outcome | relapse, complete remission |

**Relationship types** (directed edges):
- `TREATS` — drug or regimen treats a disease/stage
- `CAUSES` — drug causes a side effect
- `INDICATES` — finding indicates a disease or stage
- `REQUIRES` — stage requires a procedure
- `ASSOCIATED_WITH` — general association
- `PART_OF` — component or sub-concept relationship

Using `Literal[...]` in Pydantic + `.with_structured_output(GraphExtraction)` forces the LLM to return valid typed JSON — every extraction is **deterministic and parseable**.

In [ ]:
from pydantic import BaseModel
from typing import List, Literal

class Entity(BaseModel):
    name: str
    type: Literal["Disease", "Stage", "Drug", "SideEffect", "Procedure", "Histology", "Outcome"]
    description: str

class Relationship(BaseModel):
    source: str
    target: str
    type: Literal["TREATS", "CAUSES", "INDICATES", "REQUIRES", "ASSOCIATED_WITH", "PART_OF"]
    description: str

class GraphExtraction(BaseModel):
    entities: List[Entity]
    relationships: List[Relationship]

## Cell 2 — Load WT.pdf chunks (treatment/staging section)

We work with a **small subset (~8 chunks)** for two reasons:
1. **Cost control** — each chunk triggers one LLM API call for graph extraction
2. **Coherence** — 1500-char chunks with 150-char overlap preserve complete guideline sections

The keyword filter (`stage`, `chemotherapy`, `vincristine`, etc.) targets the *treatment protocol* pages where entity density is highest. Processing the full document adds noise (patient-info pages, bibliography) without improving the graph.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

MD_PATH = '/content/wt.md'

if not os.path.exists(MD_PATH):
    # ── Option 1: Load WT_parsed.md from Google Drive ────────────────────────────────────────────
    try:
        from google.colab import drive
        import shutil
        drive.mount('/content/drive')
        shutil.copy('/content/drive/MyDrive/CCI_Session6/WT_parsed.md', MD_PATH)
        print(f"Loaded WT_parsed.md from Google Drive → {MD_PATH}")
    except Exception as _e:
        print(f"Drive unavailable ({_e}), trying next option...")

if not os.path.exists(MD_PATH):
    # ── Option 2: Upload WT_parsed.md from your PC ───────────────────────────────────────────────
    # from google.colab import files
    # import shutil
    # files.upload()  # pick WT_parsed.md
    # shutil.copy('WT_parsed.md', MD_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────

    # ── Option 3: Re-parse WT.pdf (needs LLAMA_CLOUD_API_KEY + WT.pdf uploaded) ──────────────────
    from llama_parse import LlamaParse
    docs = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data('/content/WT.pdf')
    with open(MD_PATH, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(d.text for d in docs))
    print(f"Re-parsed and saved → {MD_PATH}")

md_text = open(MD_PATH, encoding='utf-8').read()

keywords = ['stage', 'treatment', 'chemotherapy', 'regimen', 'vincristine', 'dactinomycin', 'doxorubicin', 'side effect', 'histology']
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
all_chunks = splitter.split_text(md_text)
chunks = [c for c in all_chunks if any(k in c.lower() for k in keywords)][:8]
print(f'{len(chunks)} chunks selected')

## Cell 3 — Extract graph with LLM

For each text chunk we call `chain.invoke({'text': chunk})` and get back a `GraphExtraction` object containing `.entities` and `.relationships`.

Key design decisions:
- **`temperature=0`** — deterministic extraction; same chunk produces the same entities every run
- **Canonical names** — the prompt asks for lowercase entity names (`'stage iii'` not `'Stage III'`) so deduplication works correctly
- **Entity deduplication** — keyed by lowercased name; first occurrence wins (descriptions may vary slightly across chunks)
- **Relationships** — not deduplicated; duplicate edges overwrite safely in a `DiGraph`

Expect ~3–10 entities and ~3–8 relationships per chunk depending on how protocol-dense the section is.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(GraphExtraction)

system = (
    "You are a clinical knowledge-graph extractor for pediatric oncology. "
    "Extract entities (Disease, Stage, Drug, SideEffect, Procedure, Histology, Outcome) "
    "and relationships (TREATS, CAUSES, INDICATES, REQUIRES, ASSOCIATED_WITH, PART_OF) from the text. "
    "Use canonical, lowercase entity names (e.g., 'cisplatin', 'stage iii', 'wilms tumor'). "
    "Only include relationships explicitly supported by the text."
)
prompt = ChatPromptTemplate.from_messages([('system', system), ('human', '{text}')])
chain = prompt | llm

all_entities = {}
all_relationships = []
for i, ch in enumerate(chunks):
    try:
        ex: GraphExtraction = chain.invoke({'text': ch})
        for e in ex.entities:
            key = e.name.lower().strip()
            if key not in all_entities:
                all_entities[key] = e
        for r in ex.relationships:
            r.source = r.source.lower().strip()
            r.target = r.target.lower().strip()
            all_relationships.append(r)
        print(f'Chunk {i}: +{len(ex.entities)} ent, +{len(ex.relationships)} rel')
    except Exception as e:
        print(f'Chunk {i} failed: {e}')

print(f'\nTotal entities: {len(all_entities)} | relationships: {len(all_relationships)}')

## Cell 4 — Build NetworkX graph

We assemble the extracted entities and relationships into a `nx.DiGraph` (directed graph).

- **Nodes** = entities, with `type` and `description` as node attributes
- **Edges** = relationships, with `type` (e.g. `TREATS`) and `description` as edge attributes
- The `if r.source in G and r.target in G` guard silently drops relationships where either endpoint wasn't extracted — avoids phantom nodes

**Top-degree hubs** reveal the most-connected clinical concepts in the guideline. Expect `wilms tumor`, `stage iii`, and `vincristine` to be highly connected.

In [ ]:
import networkx as nx
G = nx.DiGraph()
for name, e in all_entities.items():
    G.add_node(name, type=e.type, description=e.description)
for r in all_relationships:
    if r.source in G and r.target in G:
        G.add_edge(r.source, r.target, type=r.type, description=r.description)

print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')
top = sorted(G.degree, key=lambda x: -x[1])[:10]
print('Top-degree:')
for n, d in top:
    print(f'  {n} ({G.nodes[n].get("type")}) deg={d}')

## Cell 5 — Visualize with pyvis

[pyvis](https://pyvis.readthedocs.io) renders interactive force-directed graphs inside Colab using D3.js.

- Each **node** is colored by entity type and shows a tooltip (type + description) on hover
- Each **edge** is labeled with its relationship type
- `cdn_resources='in_line'` embeds the JS/CSS directly in the HTML so it works offline and in Colab
- `net.generate_html()` + `display(HTML(...))` is the reliable Colab rendering path — `net.show()` is deprecated in pyvis ≥0.7.0

Zoom, drag, and click nodes to explore clinical relationships.

In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML

color_map = {
    'Disease':   '#e74c3c',  # red
    'Drug':      '#3498db',  # blue
    'Stage':     '#9b59b6',  # purple
    'SideEffect':'#f39c12',  # orange
    'Procedure': '#1abc9c',  # teal
    'Histology': '#2ecc71',  # green
    'Outcome':   '#34495e',  # dark grey
}

net = Network(height='600px', width='100%', directed=True, cdn_resources='in_line')
for n, data in G.nodes(data=True):
    net.add_node(n, label=n,
                 color=color_map.get(data.get('type'), '#888888'),
                 title=f"{data.get('type')}: {data.get('description','')}")
for s, t, d in G.edges(data=True):
    net.add_edge(s, t, label=d.get('type', ''), title=d.get('description', ''))

display(HTML(net.generate_html()))

## Cell 6 — Multi-hop graph query

**Question:** *"What drugs treat Stage III Wilms tumor and what are their side effects?"*

This is a **2-hop traversal**:
```
Stage III  ←[TREATS]―  Drug  ―[CAUSES]→  SideEffect
```
Vector RAG can answer this if the drug–stage and drug–toxicity facts happen to land in the same retrieved chunk. But if they're in different chunks, vector similarity may miss the link. The graph makes this path **explicit and queryable**.

The function handles LLM extraction inconsistencies by checking edges in **both directions** for `TREATS` (the model sometimes writes `drug TREATS stage` and sometimes `stage TREATS drug`).

In [ ]:
def multi_hop_query(G, stage_query='stage iii'):
    # 1. Find candidate stage node
    candidates = [n for n in G.nodes if stage_query in n.lower() and G.nodes[n].get('type') == 'Stage']
    if not candidates:
        candidates = [n for n in G.nodes if stage_query in n.lower()]
    if not candidates:
        return {'error': f'No node found for {stage_query}'}
    stage = candidates[0]

    # 2. Drugs that TREAT this stage (drug --TREATS--> stage)
    drugs = []
    for u, v, d in G.in_edges(stage, data=True):
        if d.get('type') == 'TREATS' and G.nodes[u].get('type') == 'Drug':
            drugs.append(u)
    # also check outgoing in case extractor reversed
    for u, v, d in G.out_edges(stage, data=True):
        if d.get('type') == 'TREATS' and G.nodes[v].get('type') == 'Drug':
            drugs.append(v)

    # 3. Side effects of those drugs (drug --CAUSES--> sideeffect)
    drug_to_se = {}
    for drug in drugs:
        ses = [v for u, v, d in G.out_edges(drug, data=True) if d.get('type') == 'CAUSES']
        drug_to_se[drug] = ses

    return {'stage': stage, 'drugs': drugs, 'drug_to_side_effects': drug_to_se}

import json
print(json.dumps(multi_hop_query(G), indent=2))

## Cell 7 — Community detection

[Louvain community detection](https://en.wikipedia.org/wiki/Louvain_method) partitions the graph into clusters of densely-connected nodes. Each community is a *thematic module* — for example, one community might contain all staging nodes + their treatment drugs, while another groups toxicity-related entities.

We convert to an undirected graph first because Louvain works on undirected edges. `seed=42` makes the result reproducible (Louvain is randomized internally).

In [ ]:
import networkx.algorithms.community as nx_comm

UG = G.to_undirected()
communities = nx_comm.louvain_communities(UG, seed=42)
for i, c in enumerate(communities):
    print(f'\n=== Theme {i} ({len(c)} members) ===')
    types = [G.nodes[n].get('type', '?') for n in c]
    print('  types:', dict((t, types.count(t)) for t in set(types)))
    print('  members:', sorted(c))

---
## Cell 8 — Vector RAG vs graph (when each wins)

**Vector RAG** (FAISS here): retrieves chunks whose *embedding* is close to the question. Strong for paraphrase and “needle in a haystack” facts **if** the fact and question live in the same chunk.

**Graph RAG**: encodes **typed edges** (e.g. Drug —TREATS→ Stage). Multi-hop questions (drug → indication → toxicity) are **path queries**, not similarity luck.

Hybrid systems often use **vectors for recall** + **graph for verification or reasoning**.


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI as Chat
from langchain_community.vectorstores import FAISS

if not chunks:
    print("⚠️  `chunks` is empty — run Cell 2 first to load wt.md and filter text chunks.")
else:
    emb = OpenAIEmbeddings(model='text-embedding-3-small')
    vs = FAISS.from_texts(chunks, emb)
    question = 'What drugs treat Stage III Wilms tumor and what are their side effects?'
    docs = vs.similarity_search(question, k=4)

    ctx = '\n---\n'.join(d.page_content for d in docs)
    answer = Chat(model='gpt-4o-mini', temperature=0).invoke(
        f"Answer using only this context.\n\nContext:\n{ctx}\n\nQuestion: {question}"
    ).content
    print('VECTOR RAG ANSWER:\n', answer)
    print('\nGRAPH RAG ANSWER:\n', json.dumps(multi_hop_query(G), indent=2))
    print('\nObservation: Vector RAG pulls semantically similar chunks. If side-effect info lives in a different chunk than the regimen info, the link can be missed. The graph encodes (Drug)-[CAUSES]->(SideEffect) explicitly.')

## Stretch — NL to graph query

In [ ]:
from pydantic import BaseModel
from typing import List as L, Literal as Lt

class Hop(BaseModel):
    edge_type: Lt['TREATS', 'CAUSES', 'INDICATES', 'REQUIRES', 'ASSOCIATED_WITH', 'PART_OF']
    direction: Lt['out', 'in']

class Plan(BaseModel):
    start_node_query: str
    hops: L[Hop]

planner = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(Plan)
plan = planner.invoke(f"Translate this question into a graph traversal plan over a graph with edges TREATS, CAUSES, INDICATES, REQUIRES, ASSOCIATED_WITH, PART_OF: {question}")
print(plan)

def execute_plan(G, plan: Plan):
    starts = [n for n in G.nodes if plan.start_node_query.lower() in n.lower()]
    frontier = set(starts)
    trace = [list(frontier)]
    for hop in plan.hops:
        nxt = set()
        for n in frontier:
            edges = G.out_edges(n, data=True) if hop.direction == 'out' else G.in_edges(n, data=True)
            for u, v, d in edges:
                if d.get('type') == hop.edge_type:
                    nxt.add(v if hop.direction == 'out' else u)
        frontier = nxt
        trace.append(list(frontier))
    return trace

print(execute_plan(G, plan))

## KHCC connection

A KHCC oncology knowledge graph could link tumor boards, regimens, biomarkers, and outcomes — enabling multi-hop questions like *"Which Stage III favorable-histology patients on Regimen DD-4A had relapse and what salvage was used?"* — the kind of reasoning vector RAG cannot do alone.